# Modeling 

In [28]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,KFold, cross_val_score,RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import make_scorer, mean_absolute_error,r2_score

from xgboost import XGBRegressor
from scipy.stats import randint, uniform

In [30]:
# load data
df = pd.read_csv('../data/clean/5_beforward_clean_data.csv')
df

,make,model,model_code,year,car_age,engine_cc,engine_code,fuel,transmission_type,drive,...,doors,price_usd,total_price_usd,shipping_cost,location,destination_port,accessories,vehicle_id,ref_no,vehicle_url
0,JEEP,WRANGLER UNLIMITED SAHARA,ABA-JL36L,2019,7,3600.0,G,Petrol,Automatic,4WD,...,5.0,16980.0,19896.0,2916.0,Yokohama,Mombasa,[''],15018558,CD066265,/jeep/wrangler/cd066265/id/15018558/?mfg_year_...
1,HONDA,ACCORD,NaN,2019,7,1500.0,NaN,Petrol,Automatic,2WD,...,4.0,27740.0,29731.0,1991.0,Thailand,Mombasa,"['Power Steering', 'A/C', 'Airbag', 'Leather S...",14864798,CC842281,/honda/accord/cc842281/id/14864798/?mfg_year_f...
2,HONDA,CITY,NaN,2024,2,1000.0,NaN,Petrol,Automatic,2WD,...,4.0,17190.0,18416.0,1226.0,Thailand,Mombasa,"['Power Steering', 'A/C', 'Airbag', 'Leather S...",14864891,CC842374,/honda/city/cc842374/id/14864891/?mfg_year_fro...
3,HONDA,BR-V,NaN,2024,2,1500.0,NaN,Petrol,Automatic,2WD,...,5.0,22080.0,24107.0,2027.0,Thailand,Mombasa,"['Power Steering', 'A/C', 'Airbag', 'Leather S...",14864903,CC842386,/honda/br-v/cc842386/id/14864903/?mfg_year_fro...
4,HONDA,CR-V,NaN,2023,3,1500.0,NaN,Petrol,Automatic,4WD,...,5.0,40390.0,42560.0,2170.0,Thailand,Mombasa,"['Power Steering', 'A/C', 'Airbag', 'Leather S...",14864910,CC842393,/honda/cr-v/cc842393/id/14864910/?mfg_year_fro...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24770,ISUZU,D-MAX 3.0 SX DOUBLE CAB,SX,2026,0,3000.0,NaN,Diesel,Automatic,4WD,...,4.0,41260.0,46045.0,4785.0,Australia,Mombasa,"['Power Steering', 'A/C', 'Airbag', 'Leather S...",15172794,CD219642,/isuzu/d-max/cd219642/id/15172794/?mfg_year_fr...
24771,ISUZU,D-MAX 3.0 LS-U+ DOUBLE CAB,LS-U,2026,0,3000.0,NaN,Diesel,Automatic,4WD,...,4.0,50090.0,54775.0,4685.0,Australia,Mombasa,"['Power Steering', 'A/C', 'Airbag', 'Leather S...",15172792,CD219641,/isuzu/d-max/cd219641/id/15172792/?mfg_year_fr...
24772,ISUZU,D-MAX 3.0 LS-U+ DOUBLE CAB,LS-U,2026,0,3000.0,NaN,Diesel,Automatic,4WD,...,4.0,54520.0,59205.0,4685.0,Australia,Mombasa,"['Power Steering', 'A/C', 'Airbag', 'Leather S...",15172791,CD219640,/isuzu/d-max/cd219640/id/15172791/?mfg_year_fr...
24773,ISUZU,D-MAX 3.0 LS-U+ DOUBLE CAB,LS-U,2025,1,3000.0,NaN,Diesel,Automatic,4WD,...,4.0,51910.0,56595.0,4685.0,Australia,Mombasa,"['Power Steering', 'A/C', 'Airbag', 'Leather S...",15172790,CD219639,/isuzu/d-max/cd219639/id/15172790/?mfg_year_fr...


In [3]:
# drop unneccesary columns
# df = df.drop(columns=['engine_code'])

In [31]:
# drop target null value
df = df.dropna(subset=["price_usd"])

In [32]:
df.info()

<class 'pandas.DataFrame'>
Index: 24744 entries, 0 to 24774
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   make               24744 non-null  str    
 1   model              24744 non-null  str    
 2   model_code         22087 non-null  str    
 3   year               24744 non-null  int64  
 4   car_age            24744 non-null  int64  
 5   engine_cc          24521 non-null  float64
 6   engine_code        1299 non-null   str    
 7   fuel               24744 non-null  str    
 8   transmission_type  24744 non-null  str    
 9   drive              24744 non-null  str    
 10  mileage_km         23607 non-null  float64
 11  steering           24744 non-null  str    
 12  color              24744 non-null  str    
 13  seats              23870 non-null  float64
 14  doors              24535 non-null  float64
 15  price_usd          24744 non-null  float64
 16  total_price_usd    24568 non-null  flo

In [33]:
# Clip extreme outliers
q_low = df["price_usd"].quantile(0.01)
q_high = df["price_usd"].quantile(0.99)

df = df[(df["price_usd"] >= q_low) & (df["price_usd"] <= q_high)]

In [34]:
df.info()

<class 'pandas.DataFrame'>
Index: 24250 entries, 0 to 24774
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   make               24250 non-null  str    
 1   model              24250 non-null  str    
 2   model_code         21595 non-null  str    
 3   year               24250 non-null  int64  
 4   car_age            24250 non-null  int64  
 5   engine_cc          24028 non-null  float64
 6   engine_code        1154 non-null   str    
 7   fuel               24250 non-null  str    
 8   transmission_type  24250 non-null  str    
 9   drive              24250 non-null  str    
 10  mileage_km         23161 non-null  float64
 11  steering           24250 non-null  str    
 12  color              24250 non-null  str    
 13  seats              23429 non-null  float64
 14  doors              24047 non-null  float64
 15  price_usd          24250 non-null  float64
 16  total_price_usd    24078 non-null  flo

In [35]:
# target variable
TARGET = "price_usd"

In [36]:
# log transform target
df[TARGET] = np.log1p(df[TARGET])

In [37]:
# columns to remove from the features columns
# reason -> 
remove_cols = [
    "price_usd",          
    "total_price_usd",    
    "shipping_cost",      
    "vehicle_id",
    "ref_no",
    "vehicle_url",
    "engine_code"
]

In [38]:
# Separating features from target

X = df.drop(remove_cols, axis=1)
y = df[TARGET]

In [39]:
X.info()

<class 'pandas.DataFrame'>
Index: 24250 entries, 0 to 24774
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   make               24250 non-null  str    
 1   model              24250 non-null  str    
 2   model_code         21595 non-null  str    
 3   year               24250 non-null  int64  
 4   car_age            24250 non-null  int64  
 5   engine_cc          24028 non-null  float64
 6   fuel               24250 non-null  str    
 7   transmission_type  24250 non-null  str    
 8   drive              24250 non-null  str    
 9   mileage_km         23161 non-null  float64
 10  steering           24250 non-null  str    
 11  color              24250 non-null  str    
 12  seats              23429 non-null  float64
 13  doors              24047 non-null  float64
 14  location           24250 non-null  str    
 15  destination_port   24250 non-null  str    
 16  accessories        24250 non-null  str

In [40]:
y.info()

<class 'pandas.Series'>
Index: 24250 entries, 0 to 24774
Series name: price_usd
Non-Null Count  Dtype  
--------------  -----  
24250 non-null  float64
dtypes: float64(1)
memory usage: 378.9 KB


In [41]:
print(f"Features remaining: {list[X.columns]}\n")
print(f"Shape: {X.shape}\n")
print(f"The Target : {TARGET}")

Features remaining: list[Index(['make', 'model', 'model_code', 'year', 'car_age', 'engine_cc', 'fuel',
       'transmission_type', 'drive', 'mileage_km', 'steering', 'color',
       'seats', 'doors', 'location', 'destination_port', 'accessories'],
      dtype='str')]

Shape: (24250, 17)

The Target : price_usd


In [42]:
X.isnull().sum()

make                    0
model                   0
model_code           2655
year                    0
car_age                 0
engine_cc             222
fuel                    0
transmission_type       0
drive                   0
mileage_km           1089
steering                0
color                   0
seats                 821
doors                 203
location                0
destination_port        0
accessories             0
dtype: int64

In [43]:
y.isnull().sum()

np.int64(0)

In [44]:
# splitting then preprocessing
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y, 
                                                    test_size=0.2,
                                                    random_state=42
                                                   )
                                                
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (19400, 17) | Test: (4850, 17)


In [45]:
# Identify column types
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['str']).columns.tolist()

print(f"Numerical: {num_cols}")
print(f"Categorical: {cat_cols}")

Numerical: ['year', 'car_age', 'engine_cc', 'mileage_km', 'seats', 'doors']
Categorical: ['make', 'model', 'model_code', 'fuel', 'transmission_type', 'drive', 'steering', 'color', 'location', 'destination_port', 'accessories']


In [48]:
# Building the pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore",min_frequency=10))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [50]:
# models
models = {
    "LinearRegression": LinearRegression(),

    "RandomForest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "GradientBoosting": GradientBoostingRegressor(
        random_state=42
    ),

    "KNN": KNeighborsRegressor(),

    "XGBoost": XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
}

# crosss validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [51]:
# training pipeline

results = {}

for name, model in models.items():

    full_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    mae_scores = cross_val_score(
        full_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='neg_mean_absolute_error'
    )

    rmse_scores = cross_val_score(
        full_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='neg_root_mean_squared_error'
    )

    r2_scores = cross_val_score(
        full_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='r2'
    )

    results[name] = {
        "MAE_mean": -mae_scores.mean(),
        "MAE_std": mae_scores.std(),
        "RMSE_mean": -rmse_scores.mean(),
        "RMSE_std": rmse_scores.std(),
        "R2_mean": r2_scores.mean(),
        "R2_std": r2_scores.std(),
    }

    print(
        f"{name:<20} "
        f"MAE: {-mae_scores.mean():.3f} | "
        f"RMSE: {-rmse_scores.mean():.3f} | "
        f"R2: {r2_scores.mean():.3f}"
    )

LinearRegression     MAE: 0.186 | RMSE: 1.093 | R2: -9.459
RandomForest         MAE: 0.129 | RMSE: 0.197 | R2: 0.894
GradientBoosting     MAE: 0.181 | RMSE: 0.245 | R2: 0.837
KNN                  MAE: 0.153 | RMSE: 0.225 | R2: 0.863
XGBoost              MAE: 0.150 | RMSE: 0.211 | R2: 0.879


In [52]:
results

{'LinearRegression': {'MAE_mean': np.float64(0.18572512077781003),
  'MAE_std': np.float64(0.02833660731605047),
  'RMSE_mean': np.float64(1.0931235734571416),
  'RMSE_std': np.float64(1.6678220241264812),
  'R2_mean': np.float64(-9.459119236187519),
  'R2_std': np.float64(20.549191890763574)},
 'RandomForest': {'MAE_mean': np.float64(0.1290450227627163),
  'MAE_std': np.float64(0.0018858738525137345),
  'RMSE_mean': np.float64(0.19728803772044348),
  'RMSE_std': np.float64(0.005419079193118215),
  'R2_mean': np.float64(0.8941100148660006),
  'R2_std': np.float64(0.006914845462772455)},
 'GradientBoosting': {'MAE_mean': np.float64(0.1809411694750686),
  'MAE_std': np.float64(0.0023274188166331304),
  'RMSE_mean': np.float64(0.2447280133445015),
  'RMSE_std': np.float64(0.005906648704806016),
  'R2_mean': np.float64(0.8371449579099692),
  'R2_std': np.float64(0.008701141163497158)},
 'KNN': {'MAE_mean': np.float64(0.1534173198008592),
  'MAE_std': np.float64(0.0012958083648511893),
  'R

In [53]:
results_df = pd.DataFrame(results).T
results_df.sort_values("RMSE_mean")

,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
RandomForest,0.129045,0.001886,0.197288,0.005419,0.894110,0.006915
XGBoost,0.150171,0.001654,0.210727,0.005804,0.879227,0.007360
KNN,0.153417,0.001296,0.224513,0.004203,0.862957,0.006357
GradientBoosting,0.180941,0.002327,0.244728,0.005907,0.837145,0.008701
LinearRegression,0.185725,0.028337,1.093124,1.667822,-9.459119,20.549192


In [24]:
best_name = min(results, key=lambda k: results[k]["RMSE_mean"])
print(f"Best model is {best_name}")

Best model is XGBoost


In [25]:
best_name = max(results, key=lambda k: results[k]["R2_mean"])
print(f"Best model: {best_name}")

Best model: XGBoost


In [25]:
# random forest
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline


param_dist = {
    'model__n_estimators': randint(50, 400),
    'model__max_depth': randint(2, 25),
    'model__min_samples_split': randint(2, 15),
    'model__min_samples_leaf': randint(1, 8)
}

best_pipe_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

search_rf = RandomizedSearchCV(
    best_pipe_rf,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

### Random Forest

In [27]:
search_rf.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': <scipy.stats....x7f984b1207d0>, 'model__min_samples_leaf': <scipy.stats....x7f98575fe2c0>, 'model__min_samples_split': <scipy.stats....x7f984b121d10>, 'model__n_estimators': <scipy.stats....x7f984b224ec0>}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will 

In [28]:
# random forest metric
print("Best params:", search_rf.best_params_)
print("Best score:", -search_rf.best_score_)

rf_final_model = search_rf.best_estimator_

# predict
y_pred_log = rf_final_model.predict(X_test)

# log space evaluation
print("\nLOG SPACE")
print("MAE:", mean_absolute_error(y_test, y_pred_log))
print("R2:", r2_score(y_test, y_pred_log))

# convert back
y_pred_usd = np.expm1(y_pred_log)
y_test_usd = np.expm1(y_test)

# real-world evaluation
print("\nUSD SPACE")
print("MAE:", mean_absolute_error(y_test_usd, y_pred_usd))
print("R2:", r2_score(y_test_usd, y_pred_usd))

Best params: {'model__max_depth': 21, 'model__min_samples_leaf': 1, 'model__min_samples_split': 4, 'model__n_estimators': 356}
Best score: 0.07127464932386567

LOG SPACE
MAE: 0.041993184869685195
R2: 0.9945630916280688

USD SPACE
MAE: 568.8337522408806
R2: 0.9928021787746847


### XGBoost

In [54]:
# xgboost

param_dist = {
    'model__n_estimators': randint(100, 800),
    'model__max_depth': randint(3, 10),
    'model__learning_rate': uniform(0.01, 0.3),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.6, 0.4),
    'model__min_child_weight': randint(1, 10),
    'model__gamma': uniform(0, 0.5)
}

best_pipe_xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        random_state=42,
        objective='reg:squarederror',
        n_jobs=-1
    ))
])

search_xgb = RandomizedSearchCV(
    best_pipe_xgb,
    param_distributions=param_dist,
    n_iter=40,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

In [55]:
# xgboost
search_xgb.fit(X_train, y_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__colsample_bytree': <scipy.stats....x7f5d1ab41ae0>, 'model__gamma': <scipy.stats....x7f5d1ab41260>, 'model__learning_rate': <scipy.stats....x7f5d1ab9d5b0>, 'model__max_depth': <scipy.stats....x7f5d1ab9d810>, ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",40
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be t

In [56]:
# XGBOOST

print("Best params:", search_xgb.best_params_)
print("Best score:", -search_xgb.best_score_)

xgb_final_model = search_xgb.best_estimator_

# predict (log space)
y_pred_log = xgb_final_model.predict(X_test)

# log space metrics
print("\nLOG SPACE")
print("MAE:", mean_absolute_error(y_test, y_pred_log))
print("R2:", r2_score(y_test, y_pred_log))

# convert back to USD
y_pred_usd = np.expm1(y_pred_log)
y_test_usd = np.expm1(y_test)

# metrics
print("\nUSD SPACE")
print("MAE:", mean_absolute_error(y_test_usd, y_pred_usd))
print("R2:", r2_score(y_test_usd, y_pred_usd))

Best params: {'model__colsample_bytree': np.float64(0.7323592099410596), 'model__gamma': np.float64(0.03177917514301182), 'model__learning_rate': np.float64(0.10329469651469865), 'model__max_depth': 7, 'model__min_child_weight': 3, 'model__n_estimators': 783, 'model__subsample': np.float64(0.8550229885420852)}
Best score: 0.1814733735392037

LOG SPACE
MAE: 0.12304344816715213
R2: 0.9143302631749591

USD SPACE
MAE: 2023.496643192252
R2: 0.880242958857226


In [30]:
# log targets
y_test_log = y_test
y_pred_log = xgb_final_model.predict(X_test)

# log metrics
print("LOG SPACE")
mae_log = mean_absolute_error(y_test_log, y_pred_log)
r2_log = r2_score(y_test_log, y_pred_log)
print("MAE:", mae_log)
print("R2:", r2_log)

# convert back to USD
y_test_usd = np.expm1(y_test_log)
y_pred_usd = np.expm1(y_pred_log)

# USD metrics
print("\nUSD SPACE")
mae_usd = mean_absolute_error(y_test_usd, y_pred_usd)
r2_usd = r2_score(y_test_usd, y_pred_usd)

print("MAE:", mae_usd)
print("R2:", r2_usd)

LOG SPACE
MAE: 0.16401425246012355
R2: 0.8642404598243864

USD SPACE
MAE: 2578.9926052173982
R2: 0.8788868114314045


In [26]:
df["price_usd"].describe()

count    12142.000000
mean         9.513963
std          0.607730
min          7.979681
25%          9.105091
50%          9.476620
75%          9.938300
max         11.203420
Name: price_usd, dtype: float64

In [57]:
print("USD TRUE SAMPLE:", y_test_usd[:5])
print("\nUSD PRED SAMPLE:", y_pred_usd[:5])

USD TRUE SAMPLE: 13723    10270.0
18289    18150.0
16207    15900.0
16215    35100.0
18034    21860.0
Name: price_usd, dtype: float64

USD PRED SAMPLE: [11655.817 16111.264 18388.889 31110.598 23238.371]


In [121]:
y_pred_log[:5]

array([2.3540246, 2.4060333, 2.3894374, 2.338175 , 2.2500432],
      dtype=float32)

In [122]:
y_pred_usd[:5]

array([ 9.527856 , 10.089883 ,  9.907356 ,  9.3623085,  8.488145 ],
      dtype=float32)

## Save Model

In [38]:
# random forest
from joblib import dump

dump(rf_final_model,"../models/rf_cars_from_japan_model_v1.pkl")

['../models/rf_cars_from_japan_model_v1.pkl']

In [39]:
# xgboost
dump(xgb_final_model, "../models/xgboost_cars_from_japan_model_v1.pkl")

['../models/xgboost_cars_from_japan_model_v1.pkl']